# DarkSight — E5 and E6: explaining the result

Two follow-up experiments that turn "brightening helped / did not help" into *why*.

**E5 — brighten only the photos that are actually dark.**
Zero-DCE brightens everything it is given, including photos that were already well lit, which can wash
them out. Here we measure each photo's average brightness and brighten it **only if it is below a
threshold τ**. τ is chosen on the validation set, never on the test set.

**E6 — swap Zero-DCE for CLAHE.**
CLAHE is a classical contrast-stretching method from the 1980s, no neural network. If CLAHE does just as
well, the deep enhancer is not earning its place — an honest check your professor will likely ask about.

### Settings
**GPU T4 x2** · **Internet On** → **Run All** (~40 min, no training).

### Required
Attach notebook 06's output (**Add Input → Your Work → Notebook Output**) so `det_enh.pt` is available;
attach notebook 05's output too if you want `det_raw.pt` for E6.

## Step 0 · Setup (same as before)

In [ ]:
!pip -q install ultralytics gdown

import os, shutil, time, json
from pathlib import Path
from collections import Counter
import numpy as np, torch, cv2, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

START = time.time()
WORK, TEMP = Path('/kaggle/working'), Path('/kaggle/temp')
DEVICE = [0, 1]            # both T4s. Change to 0 if multi-GPU training misbehaves.
MAX_SIDE = 1280

CLASSES = ["Bicycle", "Boat", "Bottle", "Bus", "Car", "Cat",
           "Chair", "Cup", "Dog", "Motorbike", "People", "Table"]
CLASS_ID = {n.lower(): i for i, n in enumerate(CLASSES)}
print('GPUs:', torch.cuda.device_count())

In [ ]:
# Two small files from our repo: the official split list and the Zero-DCE weights.
!git clone -q --depth 1 https://github.com/Manideep-21/darksight.git {TEMP}/repo
SPLIT_FILE, ZERODCE_FILE = TEMP/'repo/configs/exdark_imageclasslist.txt', TEMP/'repo/weights/zerodce_epoch99.pth'

# ExDark itself (1.5 GB photos + 5 MB boxes) from the authors' Google Drive.
SRC = TEMP / 'exdark_src'
if not SRC.exists():
    !gdown -q 1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx -O {TEMP}/images.zip
    !gdown -q 1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i -O {TEMP}/boxes.zip
    !mkdir -p {SRC} && unzip -q -o {TEMP}/images.zip -d {SRC} && unzip -q -o {TEMP}/boxes.zip -d {SRC}
    !rm -rf {SRC}/__MACOSX

photos = {p.stem.lower(): p for p in (SRC/'ExDark').rglob('*') if p.is_file() and not p.name.startswith('.')}
boxes  = {p.name.lower().removesuffix('.txt').rsplit('.', 1)[0]: p for p in (SRC/'ExDark_Annno').rglob('*.txt')}

SPLIT_NAME = {'1': 'train', '2': 'val', '3': 'test'}
LIGHTING = {'1':'Low','2':'Ambient','3':'Object','4':'Single','5':'Weak',
            '6':'Strong','7':'Screen','8':'Window','9':'Shadow','10':'Twilight'}
split_of, lighting_of = {}, {}
for line in SPLIT_FILE.read_text().splitlines():
    c = line.split()
    if len(c) == 5 and c[1].isdigit():
        k = Path(c[0]).stem.lower()
        split_of[k], lighting_of[k] = SPLIT_NAME[c[4]], LIGHTING[c[2]]
print(len(photos), 'photos |', Counter(split_of.values()))

In [ ]:
def convert_one(photo_path, box_path, out_photo, out_label):
    """Save one photo as PNG and its boxes as a YOLO label file (centre-x, centre-y, w, h as fractions)."""
    img = Image.open(photo_path).convert('RGB')
    W, H = img.size
    lines = []
    for line in Path(box_path).read_text(errors='ignore').splitlines():
        parts = line.split()
        if not parts or parts[0].startswith('%'):
            continue
        name, l, t, w, h = parts[0], *map(float, parts[1:5])
        cls = CLASS_ID.get(name.lower())
        if cls is None:
            continue
        x1, y1 = max(l, 0), max(t, 0)
        x2, y2 = min(l + w, W), min(t + h, H)
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        lines.append(f"{cls} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    if max(W, H) > MAX_SIDE:
        img.thumbnail((MAX_SIDE, MAX_SIDE))
    out_photo.parent.mkdir(parents=True, exist_ok=True)
    out_label.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_photo)
    out_label.write_text("\n".join(lines))

RAW = TEMP / 'data/raw'
for i, (key, photo) in enumerate(sorted(photos.items())):
    convert_one(photo, boxes[key], RAW/'images'/split_of[key]/f'{key}.png',
                RAW/'labels'/split_of[key]/f'{key}.txt')
    if i % 1500 == 0:
        print(f'{i}/{len(photos)}')
print('converted', len(photos), 'photos')

In [ ]:
import torch.nn as nn

class ZeroDCE(nn.Module):
    """Predicts 8 brightening curves; each is applied as  new = old + a*old*(1-old)."""
    def __init__(self, f=32):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.e_conv1 = nn.Conv2d(3, f, 3, 1, 1);     self.e_conv2 = nn.Conv2d(f, f, 3, 1, 1)
        self.e_conv3 = nn.Conv2d(f, f, 3, 1, 1);     self.e_conv4 = nn.Conv2d(f, f, 3, 1, 1)
        self.e_conv5 = nn.Conv2d(f*2, f, 3, 1, 1);   self.e_conv6 = nn.Conv2d(f*2, f, 3, 1, 1)
        self.e_conv7 = nn.Conv2d(f*2, 24, 3, 1, 1)
    def forward(self, x):
        x1 = self.relu(self.e_conv1(x));  x2 = self.relu(self.e_conv2(x1))
        x3 = self.relu(self.e_conv3(x2)); x4 = self.relu(self.e_conv4(x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1)))
        for r in torch.split(torch.tanh(self.e_conv7(torch.cat([x1, x6], 1))), 3, dim=1):
            x = x + r * (x * x - x)
        return x

zerodce = ZeroDCE().cuda().eval()
zerodce.load_state_dict(torch.load(ZERODCE_FILE, map_location='cuda'))

@torch.no_grad()
def brighten(img):
    x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().cuda() / 255.0
    return (zerodce(x).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')

In [ ]:
def write_yaml(path, train_dir, val_dir, test_dir):
    """Tell YOLO where the photos are. Absolute paths, so 'path:' is just the filesystem root."""
    names = "\n".join(f'  {i}: {n}' for i, n in enumerate(CLASSES))
    Path(path).write_text(f"path: /\ntrain: {train_dir}\nval: {val_dir}\ntest: {test_dir}\nnames:\n{names}\n")
    return str(path)

def draw_boxes(img, boxes_xyxy, labels):
    out = img.copy()
    for (x1, y1, x2, y2), text in zip(boxes_xyxy, labels):
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(out, text, (int(x1), max(int(y1)-5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return out

def find_in_inputs(filename):
    """Look for a file inside any attached Kaggle dataset / notebook output."""
    hits = sorted(Path('/kaggle/input').rglob(filename)) if Path('/kaggle/input').exists() else []
    return hits[0] if hits else None

In [ ]:
from ultralytics import YOLO

det_enh_path = find_in_inputs('det_enh.pt')
det_raw_path = find_in_inputs('det_raw.pt')
assert det_enh_path, 'Attach notebook 06 output (det_enh.pt) via Add Input -> Your Work -> Notebook Output'
m_enh = YOLO(det_enh_path)
m_raw = YOLO(det_raw_path) if det_raw_path else None
print('det_enh:', det_enh_path, '| det_raw:', det_raw_path or 'not attached (E6 will use det_enh)')

## Step 1 · Brighten the val and test photos (~5 min)

In [ ]:
ENH = TEMP / 'data/enhanced'
for split in ['val', 'test']:
    shutil.rmtree(ENH/'labels'/split, ignore_errors=True)
    shutil.copytree(RAW/'labels'/split, ENH/'labels'/split)
    (ENH/'images'/split).mkdir(parents=True, exist_ok=True)
    files = sorted((RAW/'images'/split).glob('*.png'))
    for i, p in enumerate(files):
        Image.fromarray(brighten(np.array(Image.open(p)))).save(ENH/'images'/split/p.name)
        if i % 800 == 0: print(f'{split}: {i}/{len(files)}')
print('done')

## Step 2 · How bright is each photo?

Average brightness, using the standard formula for how bright a colour looks to the eye
(green counts most, blue least). 0 = black, 1 = white.

In [ ]:
def brightness(img):
    rgb = img.astype(np.float32) / 255.0
    return float((0.299*rgb[...,0] + 0.587*rgb[...,1] + 0.114*rgb[...,2]).mean())

bright_of = {}
for split in ['val', 'test']:
    for p in sorted((RAW/'images'/split).glob('*.png')):
        bright_of[p.stem] = brightness(np.array(Image.open(p)))

vals = np.array([bright_of[k] for k in bright_of if split_of[k] == 'val'])
print(f'validation photos: darkest {vals.min():.3f}, median {np.median(vals):.3f}, brightest {vals.max():.3f}')
plt.figure(figsize=(8, 3)); plt.hist(vals, bins=40)
plt.xlabel('average brightness'); plt.ylabel('photos'); plt.title('How dark is ExDark really?')
plt.show()

## Step 3 · E5 — pick τ on the validation set

A "gated" set is built by picking, for each photo, either the brightened copy (if the original was darker
than τ) or the original. Nothing is recomputed, we just choose files — so trying several τ values is cheap.

In [ ]:
def build_gated(split, tau, out_dir):
    """Brightened copy if the photo is darker than tau, otherwise the original."""
    shutil.rmtree(out_dir, ignore_errors=True)
    (out_dir/'images'/split).mkdir(parents=True, exist_ok=True)
    shutil.copytree(RAW/'labels'/split, out_dir/'labels'/split)
    n_brightened = 0
    for p in sorted((RAW/'images'/split).glob('*.png')):
        source = ENH if bright_of[p.stem] < tau else RAW
        n_brightened += source is ENH
        shutil.copy(source/'images'/split/p.name, out_dir/'images'/split/p.name)
    return n_brightened

TAUS = [1.00, 0.45, 0.35, 0.25, 0.15]      # 1.00 = brighten everything (= plain E4 behaviour)
val_scores = []
for tau in TAUS:
    out = TEMP/f'gated_val_{tau}'
    n = build_gated('val', tau, out)
    y = write_yaml(TEMP/f'gval_{tau}.yaml', RAW/'images/train', out/'images/val', out/'images/val')
    score = m_enh.val(data=y, split='val', imgsz=640, device=0, verbose=False).box.map50
    val_scores.append({'tau': tau, 'brightened': n, 'val mAP@0.5': round(score, 4)})
    print(val_scores[-1])

val_table = pd.DataFrame(val_scores)
best_tau = float(val_table.loc[val_table['val mAP@0.5'].idxmax(), 'tau'])
print(f'\nbest tau on validation = {best_tau}')
val_table.to_csv(WORK/'E5_tau_search_val.csv', index=False)

In [ ]:
# Apply that tau to the test set - this is E5. The test set is touched once, with tau already decided.
n = build_gated('test', best_tau, TEMP/'gated_test')
print(f'brightened {n} of 2563 test photos (tau = {best_tau})')

yaml_raw  = write_yaml(TEMP/'raw.yaml',   RAW/'images/train', RAW/'images/val', RAW/'images/test')
yaml_enh  = write_yaml(TEMP/'enh.yaml',   RAW/'images/train', RAW/'images/val', ENH/'images/test')
yaml_gate = write_yaml(TEMP/'gate.yaml',  RAW/'images/train', RAW/'images/val', TEMP/'gated_test/images/test')

scores = {}
print('=== E4: brighten everything ===')
scores['E4 brighten all'] = m_enh.val(data=yaml_enh,  split='test', imgsz=640, device=0).box.map50
print(f'=== E5: brighten only photos darker than {best_tau} ===')
scores['E5 brighten dark only'] = m_enh.val(data=yaml_gate, split='test', imgsz=640, device=0).box.map50

## Step 4 · E6 — would a classical method do the same job?

CLAHE splits the image into tiles and stretches the contrast of each one separately, with a cap so noise
is not amplified too much. It is about 15 lines of OpenCV and runs on the CPU.

In [ ]:
def clahe_brighten(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)          # separate lightness from colour
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

CLA = TEMP / 'data/clahe'
shutil.rmtree(CLA, ignore_errors=True)
shutil.copytree(RAW/'labels/test', CLA/'labels/test')
(CLA/'images/test').mkdir(parents=True, exist_ok=True)
for i, p in enumerate(sorted((RAW/'images/test').glob('*.png'))):
    Image.fromarray(clahe_brighten(np.array(Image.open(p)))).save(CLA/'images/test'/p.name)
    if i % 800 == 0: print(f'{i}/2563')

# compare the three versions of one photo
k = sorted(k for k in split_of if split_of[k] == 'test')[0]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (title, path) in zip(axes, [('raw', RAW), ('Zero-DCE', ENH), ('CLAHE', CLA)]):
    ax.imshow(Image.open(path/'images/test'/f'{k}.png')); ax.set_title(title); ax.axis('off')
plt.show()

In [ ]:
# Judge CLAHE with the detector trained on raw photos (det_raw), which is the fair partner for it:
# neither was trained on Zero-DCE output. Falls back to det_enh if det_raw was not attached.
judge = m_raw if m_raw else m_enh
yaml_cla = write_yaml(TEMP/'clahe.yaml', RAW/'images/train', RAW/'images/val', CLA/'images/test')

print('=== baseline for CLAHE: same detector on raw photos ===')
scores['raw photos (same detector)'] = judge.val(data=yaml_raw, split='test', imgsz=640, device=0).box.map50
print('=== E6: CLAHE ===')
scores['E6 CLAHE'] = judge.val(data=yaml_cla, split='test', imgsz=640, device=0).box.map50

In [ ]:
table = pd.DataFrame({'mAP@0.5': {k: round(v, 4) for k, v in scores.items()}})
print(table.to_string())
table.to_csv(WORK/'summary_E5_E6.csv')

plt.figure(figsize=(8, 4))
plt.barh(list(scores), list(scores.values()), color='#3b6ea8')
for i, v in enumerate(scores.values()):
    plt.text(v + 0.003, i, f'{v:.3f}', va='center')
plt.xlabel('mAP@0.5'); plt.title('Does smarter brightening help?'); plt.tight_layout()
plt.savefig(WORK/'E5_E6.png', dpi=150); plt.show()

print(f'''
How to read this:
  E5 vs E4  - if E5 is higher, Zero-DCE was damaging the already-bright photos,
              and brightening only the dark ones ({best_tau} threshold) fixes part of it.
  E6 vs raw - if CLAHE also helps (or hurts) by a similar amount, the benefit is not
              coming from deep learning; it is just contrast.
tau was chosen on validation photos only, so the test numbers stay honest.
''')
print(f'Total time: {(time.time()-START)/60:.0f} min')